In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# House Prices: Advanced Regression Techniques — Full Solution (RobustScaler Edition)
# Clean, fast, and Kaggle-friendly

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, mean_squared_error

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# If you're tight on time, you can temporarily comment RandomForest to speed up CV.

# ----------------------------
# 1) Paths & Data Loading
# ----------------------------
def get_data_paths():
    kaggle_base = "/kaggle/input/house-prices-advanced-regression-techniques"
    if os.path.exists(kaggle_base):
        train_path = os.path.join("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
        test_path  = os.path.join("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
    else:
        train_path = "train.csv"  # put files next to this script if local
        test_path  = "test.csv"
    return train_path, test_path

train_path, test_path = get_data_paths()
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

# ----------------------------
# 2) Target, Features
# ----------------------------
TARGET = "SalePrice"
y = np.log1p(train[TARGET].values.astype(np.float64))
X = train.drop(columns=[TARGET])
X_test = test.copy()

# Safety: make sure target is finite
assert np.isfinite(y).all(), "Target contains non-finite values after log1p."

# ----------------------------
# 3) Feature Types & Preprocessing
# ----------------------------
# Base dtype split
base_num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
base_cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Some integers are really categories; treat them as categorical
cat_like_numeric = ["MSSubClass"]  # You may also add: "MoSold", "YrSold"
cat_cols = sorted(list(set(base_cat_cols + cat_like_numeric)))
num_cols = [c for c in X.columns if c not in cat_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler(with_centering=True, with_scaling=True)),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop",
)

# ----------------------------
# 4) Models to Try (fast & solid)
# ----------------------------
models = {
    "Ridge": Ridge(alpha=10.0, random_state=42),
    "Lasso": Lasso(alpha=0.0005, max_iter=10000, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=10000, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(
        n_estimators=400, max_depth=None, min_samples_split=2, min_samples_leaf=1,
        n_jobs=-1, random_state=42
    ),
}

# RMSE scorer on log target
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

# ----------------------------
# 5) Cross-Validation
# ----------------------------
cv = KFold(n_splits=10, shuffle=True, random_state=42)

cv_results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X, y, scoring=rmse_scorer, cv=cv, n_jobs=-1)
    mean_rmse = -scores.mean()
    std_rmse =  scores.std()
    cv_results.append((name, mean_rmse, std_rmse))
    fitted_pipelines[name] = pipe
    print(f"{name:<18} | CV RMSE (log target) = {mean_rmse:.5f} ± {std_rmse:.5f}")

cv_df = pd.DataFrame(cv_results, columns=["Model", "CV_RMSE_Log", "CV_STD"])
cv_df_sorted = cv_df.sort_values(by="CV_RMSE_Log", ascending=True).reset_index(drop=True)

print("\n=== Cross-validated results (lower is better) ===")
print(cv_df_sorted)

best_model_name = cv_df_sorted.loc[0, "Model"]
print(f"\nSelected best model: {best_model_name}")

# ----------------------------
# 6) Fit Best Model & Predict
# ----------------------------
best_pipe = fitted_pipelines[best_model_name]
best_pipe.fit(X, y)

# Predict on test (log space), then invert
y_test_pred_log = best_pipe.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)
y_test_pred = np.maximum(y_test_pred, 0)  # guard (shouldn't be needed)

# ----------------------------
# 7) Build Submission
# ----------------------------
submission = pd.DataFrame({
    "Id": X_test["Id"],
    "SalePrice": y_test_pred
})

save_path = "submission.csv"
submission.to_csv(save_path, index=False)
print(f"\nSaved file: {save_path}")
print(submission.head())


In [ ]:
# --- Produce predictions with the best model and save submission ---

# 1) Pick best model from CV table
best_model_name = cv_df_sorted.loc[0, "Model"]
best_pipe = fitted_pipelines[best_model_name]
print(f"Using best model: {best_model_name}")

# 2) Fit on the full training data (log target)
best_pipe.fit(X, y)

# 3) Predict on test set (model outputs log SalePrice), then invert the transform
y_test_pred_log = best_pipe.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)

# 4) Guard (rare): replace negatives with 0
y_test_pred = np.maximum(y_test_pred, 0)

# 5) Build and save submission
submission = pd.DataFrame({
    "Id": X_test["Id"],
    "SalePrice": y_test_pred
})

submission.to_csv("submission.csv", index=False)
print("Saved: submission.csv")
display(submission.head(10))


In [ ]:
# --- Model performance summary + bar chart ---
import matplotlib.pyplot as plt

# Display results table
display(cv_df_sorted.style.format({"CV_RMSE_Log": "{:.5f}", "CV_STD": "{:.5f}"}))

# Bar chart of CV results with error bars
plt.figure(figsize=(8,4.5))
plt.bar(cv_df_sorted["Model"], cv_df_sorted["CV_RMSE_Log"], 
        yerr=cv_df_sorted["CV_STD"], capsize=4)
plt.title("10-fold CV RMSE (log target) by Model")
plt.ylabel("RMSE (lower is better)")
plt.xlabel("Model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# --- Fit best model, predict, and save submission.csv ---
best_model_name = cv_df_sorted.loc[0, "Model"]
best_pipe = fitted_pipelines[best_model_name]
print(f"Using best model: {best_model_name}")

best_pipe.fit(X, y)

y_test_pred_log = best_pipe.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)
y_test_pred = np.maximum(y_test_pred, 0)

submission = pd.DataFrame({"Id": X_test["Id"], "SalePrice": y_test_pred})
submission.to_csv("submission.csv", index=False)
print("Saved: submission.csv")
display(submission.head(10))


In [ ]:
# --- Diagnostics using out-of-fold (OOF) predictions ---
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error

# OOF predictions in log space, then invert
y_oof_log = cross_val_predict(best_pipe, X, y, cv=cv, n_jobs=-1)
rmse_oof = np.sqrt(mean_squared_error(y, y_oof_log))
print(f"OOF RMSE (log target): {rmse_oof:.5f}")

y_true = np.expm1(y)
y_pred = np.expm1(y_oof_log)
residuals = y_true - y_pred

# Predicted vs Actual (original scale)
plt.figure(figsize=(6,5))
plt.scatter(y_true, y_pred, s=12, alpha=0.5)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
plt.plot(lims, lims)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice (OOF)")
plt.title(f"Predicted vs Actual — {best_model_name}")
plt.tight_layout()
plt.show()

# Residuals histogram (original scale)
plt.figure(figsize=(6,4))
plt.hist(residuals, bins=30)
plt.title(f"Residuals Distribution — {best_model_name}")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# --- Feature name extraction from your ColumnTransformer ---
import numpy as np

def get_feature_names(preprocess):
    """Extract final column names after ColumnTransformer+OneHot."""
    # Assumes preprocess has two transformers: ("num", ... num_cols) and ("cat", pipeline->onehot, cat_cols)
    num_names = preprocess.transformers_[0][2]
    cat_pipeline = preprocess.transformers_[1][1]
    cat_cols = preprocess.transformers_[1][2]
    onehot = cat_pipeline.named_steps["onehot"]
    cat_names = onehot.get_feature_names_out(cat_cols)
    return np.r_[num_names, cat_names]

def plot_importance(pipe, top_k=10, title="Top Feature Importances"):
    model = pipe.named_steps["model"]
    preprocess = pipe.named_steps["prep"]
    feat_names = get_feature_names(preprocess)

    if hasattr(model, "feature_importances_"):
        vals = model.feature_importances_
        label = "feature_importances_"
    elif hasattr(model, "coef_"):
        # L1/L2 models: magnitude of coefficients
        coef = model.coef_
        coef = coef.ravel() if hasattr(coef, "ravel") else coef
        vals = np.abs(coef)
        label = "|coefficients|"
    else:
        print("This model doesn't expose feature importances/coefficients.")
        return

    df = pd.DataFrame({"Feature": feat_names, "Importance": vals})
    df = df.sort_values("Importance", ascending=False).head(top_k)

    plt.figure(figsize=(8,5))
    plt.barh(df["Feature"], df["Importance"])
    plt.gca().invert_yaxis()
    plt.title(f"{title} ({best_model_name}, {label})")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

# Plot after the best_pipe is fit (done above)
plot_importance(best_pipe, top_k=10, title="Top 10 Features")


In [ ]:
# --- Robust Permutation Importance (handles Pipeline vs transformed matrix) ---
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Split a small validation set for speed
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit best pipeline
best_model_name = cv_df_sorted.loc[0, "Model"]
best_pipe = fitted_pipelines[best_model_name]
best_pipe.fit(X_tr, y_tr)

prep = best_pipe.named_steps["prep"]
model = best_pipe.named_steps["model"]

def safe_feature_names_from_preprocess(prep):
    # Preferred (sklearn >=1.0)
    if hasattr(prep, "get_feature_names_out"):
        names = prep.get_feature_names_out()
        # strip "num__"/"cat__" prefixes if present
        names = np.array([n.split("__", 1)[-1] for n in names])
        return names
    # Fallback: manually build
    names = []
    for name, trans, cols in prep.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if hasattr(trans, "get_feature_names_out"):
            sub = trans.get_feature_names_out(cols)
        elif hasattr(trans, "named_steps") and "onehot" in trans.named_steps:
            sub = trans.named_steps["onehot"].get_feature_names_out(cols)
        else:
            sub = np.asarray(cols, dtype=object)
        sub = [s.split("__", 1)[-1] if isinstance(s, str) else s for s in sub]
        names.extend(sub)
    return np.array(names, dtype=object)

# Try 1) Permutation importance on the Pipeline (one score per ORIGINAL column)
r = permutation_importance(
    best_pipe, X_val, y_val,
    n_repeats=10, random_state=42, n_jobs=-1,
    scoring="neg_root_mean_squared_error"  # RMSE-based
)

importances = r.importances_mean
names = np.array(X_val.columns)

# If lengths don't match (due to OHE expansion), fallback to model-only on transformed matrix
if importances.shape[0] != len(names):
    Xt_val = prep.transform(X_val)
    names = safe_feature_names_from_preprocess(prep)   # post-transform (OHE-expanded) names
    r = permutation_importance(
        model, Xt_val, y_val,
        n_repeats=10, random_state=42, n_jobs=-1,
        scoring="neg_root_mean_squared_error"
    )
    importances = r.importances_mean
    stds = r.importances_std
else:
    stds = r.importances_std

# Build top-10 table
perm_df = pd.DataFrame({
    "Feature": names,
    "Importance": importances,
    "Std": stds
}).sort_values("Importance", ascending=False).head(10)

display(perm_df)

# Plot
plt.figure(figsize=(8,5))
plt.barh(perm_df["Feature"], perm_df["Importance"])
plt.gca().invert_yaxis()
plt.title(f"Permutation Importance (Validation) — {best_model_name}")
plt.xlabel("Mean decrease in score (higher = more important)")
plt.tight_layout()
plt.show()


In [ ]:
# --- Partial Dependence for a few key features (works well for tree models; linear is flat) ---
from sklearn.inspection import PartialDependenceDisplay

# Pick a few top features that are numeric in your original data
key_num_feats = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt"]

# Only keep features that exist in your training columns
key_num_feats = [f for f in key_num_feats if f in X.columns]

if key_num_feats:
    fig, ax = plt.subplots(figsize=(10,6))
    PartialDependenceDisplay.from_estimator(best_pipe, X, key_num_feats, ax=ax)
    plt.suptitle(f"Partial Dependence — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No key numeric features available for partial dependence.")
